# Exercício Prático — Aula 09: limpando dados de triagem

**Análise e Ciência de Dados** (Especialização, IFRN) — Unidade 2, Aula 09

Este exercício é **para praticar** — não precisa ser entregue. A avaliação da aula é o **formulário avaliativo** no Moodle (prazo **30/10/2026**), e várias questões dele usam exatamente as situações deste exercício.

**Dados:** `dados/triagem_pa_sujo.csv` — 64 registros **sintéticos** de triagem em um pronto atendimento (nenhum paciente real): idade, temperatura, frequência cardíaca, saturação de O₂ e a classificação de risco (protocolo de cores Azul / Verde / Amarelo / Vermelho).

Em cada passo há um bloco **"Para conferir"** com o resultado esperado. Use o notebook `Aula09_Qualidade_e_Limpeza_de_Dados.ipynb` como referência.

## Passo 1 — Carregar e diagnosticar

Carregue o arquivo e responda: quantas linhas e colunas? Quantos ausentes por coluna? Quantas linhas duplicadas?

> **Para conferir:** 64 linhas × 7 colunas; 4 ausentes em `temperatura_c` e 2 em `saturacao_o2`; 4 linhas duplicadas.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/aula09_qualidade_limpeza/dados/atendimentos_ubs_sujo.csv",
                 encoding="utf-8")

print("\n### Linhas , colunas ###")
print(df.shape)
print("\n---")
print("\n### Amostra dos 6 primeiros ###")
print(df.head(6).to_string())
print("\n---")

print("### Diagnóstico ###")
ausentes = df.isna().sum()
pd.DataFrame({"ausentes": ausentes, "percentual": (ausentes / len(df) * 100).round(1)})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

## Linhas , colunas ##
(250, 10)

## Amostra dos 6 primeiros ##
   id_atendimento data_atendimento   municipio       sexo  idade  peso_kg  altura_cm  pressao_sistolica  glicemia_mg_dl         diagnostico
0            1149       2026-06-07       Natal  Masculino   86.0     85.2     171.00              133.0             NaN         Hipertensão
1            1220       2026-03-23  Parnamirim          M   41.0     90.5     171.00              131.0            89.0  Consulta de rotina
2            1095       08/05/2026  parnamirim          F   63.0     65.6     159.00              128.0           105.0         Hipertensão
3            1085       2026-06-08       Natal          M   23.0     80.8       1.59              124.0           105.0  Consulta de rotina
4            1004       2026-03-24       Natal          F   64.0     65.8     157.00              130.0   

,ausentes,percentual
id_atendimento,0,0.0
data_atendimento,0,0.0
municipio,0,0.0
sexo,6,2.4
idade,7,2.8
peso_kg,12,4.8
altura_cm,0,0.0
pressao_sistolica,6,2.4
glicemia_mg_dl,24,9.6
diagnostico,0,0.0


## Passo 2 — Remover duplicatas

> **Para conferir:** restam 60 linhas.

In [ ]:
# TODO: drop_duplicates() e reset_index(drop=True)
print("Linhas duplicadas (idênticas):", df.duplicated().sum())

print("### Dados duplicados ###")
copias = df[df.duplicated(keep=False)].sort_values("id_atendimento")
print(copias[["id_atendimento", "data_atendimento", "municipio", "idade", "diagnostico"]].head(6).to_string())

antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nRemovidas {antes - len(df)} linhas duplicadas -> {len(df)} linhas")
print("id_atendimento é único?", df["id_atendimento"].is_unique)
print()

print("\n### Agregados ###")
print(df.describe().round(1).to_string())
print("\n### Verificação de Nomeclaturas ###")
print(df["municipio"].value_counts().to_string())
print()
print(df["sexo"].value_counts(dropna=False).to_string())

Linhas duplicadas (idênticas): 0
### Dados duplicados ###
Empty DataFrame
Columns: [id_atendimento, data_atendimento, municipio, idade, diagnostico]
Index: []

Removidas 0 linhas duplicadas -> 240 linhas
id_atendimento é único? True


### Agregados ###
       id_atendimento  idade  peso_kg  altura_cm  pressao_sistolica  glicemia_mg_dl
count           240.0  233.0    229.0      240.0              234.0           218.0
mean           1120.5   54.0   1088.7      157.3              135.1           129.5
std              69.4   23.8   8974.9       31.9               80.7           136.6
min            1001.0   -3.0     35.3        1.6               12.0            62.0
25%            1060.8   37.0     60.9      156.0              119.0            91.0
50%            1120.5   53.0     72.1      162.0              128.0           101.0
75%            1180.2   70.0     84.9      170.0              141.0           115.0
max            1240.0  205.0  90900.0      185.0             1300.0        

## Passo 3 — Padronizar a classificação de risco

A coluna `classificacao_risco` tem `"verde"`, `"VERDE"`, `"Verde"`... Deixe tudo no formato `"Verde"` (primeira letra maiúscula).

> **Para conferir:** Verde 30, Amarelo 21, Azul 7, Vermelho 2.

In [ ]:
# TODO: str.strip() + str.capitalize() e depois value_counts()


## Passo 4 — Encontrar outliers com o IQR

Calcule Q1, Q3, IQR e os limites (Q1 − 1,5·IQR e Q3 + 1,5·IQR) para `freq_cardiaca`, `temperatura_c` e `saturacao_o2`, e liste os valores fora dos limites.

> **Para conferir:** frequência cardíaca → 300; temperatura → 375; saturação → 9, 90 e 92.
> Pense: quais são **erros** e quais são **valores reais** de pacientes graves?

In [ ]:
def outliers_iqr(serie):
    # TODO: devolver os valores abaixo de Q1 - 1.5*IQR ou acima de Q3 + 1.5*IQR
    pass

for col in ["freq_cardiaca", "temperatura_c", "saturacao_o2"]:
    print(col, outliers_iqr(t[col]))


## Passo 5 — Corrigir os erros (regras de domínio)

- Temperatura acima de 45 °C: a vírgula foi esquecida (375 → 37,5) → **divida por 10**.
- Frequência cardíaca fora de 30–220 bpm: impossível → **NaN**.
- Saturação abaixo de 50%: impossível → **NaN**. (90% e 92% são baixos, mas **reais** — ficam!)

> **Para conferir:** depois das correções, os ausentes passam a ser: temperatura 4, frequência cardíaca 1, saturação 3.

In [ ]:
# TODO: aplicar as três regras com df.loc[condição, coluna] = ...


## Passo 6 — Preencher os ausentes

- `temperatura_c` → mediana geral;
- `saturacao_o2` e `freq_cardiaca` → mediana **por classificação de risco** (`groupby(...).transform("median")`).

> **Para conferir:** mediana geral da temperatura = 36,8 °C; mediana da saturação no grupo Vermelho = 91%; nenhum ausente no final.

In [ ]:
# TODO: fillna com mediana geral e com mediana por grupo


## Passo 7 — Normalização e padronização

Crie `fc_minmax` (frequência cardíaca normalizada entre 0 e 1) e `temp_z` (temperatura padronizada pelo z-score).

> **Para conferir:** `fc_minmax` vai de 0 a 1 (média 0,43); `temp_z` tem média 0 e desvio-padrão 1.

In [ ]:
# TODO: fórmulas (x - min)/(max - min) e (x - média)/desvio


## Passo 8 — Conferir o resultado e salvar

Calcule as médias de temperatura, frequência cardíaca e saturação por classificação de risco e salve o arquivo limpo.

> **Para conferir:** o grupo **Vermelho** tem a maior temperatura média (38,2 °C), a maior frequência cardíaca (117 bpm) e a menor saturação (91%) — coerente com o protocolo de triagem. Se o seu resultado mostrar uma média de temperatura acima de 40 °C, algum erro ficou para trás!

In [ ]:
# TODO: groupby("classificacao_risco") + mean(); salvar em dados/triagem_pa_limpo.csv
